In [1]:
import numpy as np
from numpy import pi
import pandas as pd
import matplotlib.pyplot as plt
import time
import copy
import pickle
from ordered_set import OrderedSet

import qiskit
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager, PassManager
from qiskit.quantum_info import SparsePauliOp, Pauli
from qiskit.visualization import plot_bloch_multivector, plot_histogram

import mthree as m3
from qiskit.result import marginal_counts

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, SamplerOptions

from qiskit.primitives.containers import BitArray

<div class="alert alert-block alert-warning">
<b>Warning: Don't run this unless necessary</b>

This creates pickle data files for the required runs. Run this only if you want to change quantum circuits or observables.
</div>

# Store Required Data 

In [ ]:
# service = QiskitRuntimeService(name = "Reserach_IITJ")
# backend_list = service.backends()
# print(f"Available Backends: {backend_list}")

Available Backends: [<IBMBackend('ibm_fez')>, <IBMBackend('ibm_torino')>, <IBMBackend('ibm_marrakesh')>]


### Quantum Circuits

Unitary Gate Based Method: chain

In [3]:
# qr = QuantumRegister(8, name = "qr")
# cr = ClassicalRegister(8, name = "meas")
# ghz_qc_ugc = QuantumCircuit(qr, cr, name="GHZ_UGC")
# ghz_qc_ugc.h(4)
# for i in range(4):
#     if 4-i-1>=0:
#         ghz_qc_ugc.cx(4-i, 4-i-1)
#     if 4+i+1<8:
#         ghz_qc_ugc.cx(4+i, 4+i+1)

# ghz_qc_ugc.barrier()
# ghz_qc_ugc.measure(qr, cr)

# ghz_qc_ugc.draw('mpl')

Unitary Gate Based Method: Star

In [4]:
# qr = QuantumRegister(8, name = "qr")
# cr = ClassicalRegister(8, name = "meas")
# ghz_qc_ugs = QuantumCircuit(qr,cr, name="GHZ_UGS")
# ghz_qc_ugs.h(0)
# for i in range(1, 8):
#     ghz_qc_ugs.cx(0, i)

# ghz_qc_ugs.barrier()
# ghz_qc_ugs.measure(qr,cr)

# ghz_qc_ugs.draw('mpl')

State Initialization Method 

In [5]:
# state_0 = np.array([1,0])
# state_1 = np.array([0,1])
# kron_state_0 = state_0
# kron_state_1 = state_1
# for _ in range(7):
#     kron_state_0 = np.kron(kron_state_0, state_0)
#     kron_state_1 = np.kron(kron_state_1, state_1)
# ghz_state = (1/np.sqrt(2)) * (kron_state_0 + kron_state_1)

# qr = QuantumRegister(8, name = "qr")
# cr = ClassicalRegister(8, name = "meas")
# ghz_qc_si = QuantumCircuit(qr, cr, name="GHZ_SI")
# ghz_qc_si.initialize(ghz_state, range(8))

# ghz_qc_si.barrier()
# ghz_qc_si.measure(qr, cr)
# display(ghz_qc_si.draw('mpl', fold=-1))

Dynamic Circuits

In [6]:
# from qiskit.circuit.classical import expr

# N=8
# qr = QuantumRegister(N, name="qr")
# cr_syn = ClassicalRegister(int(N/2-1), name="cr_syn")
# cr = ClassicalRegister(N, name="meas")
# ghz_qc_dc = QuantumCircuit(qr, cr, cr_syn, name = "GHZ_DC")

# for j in range(0, N-1, 2):
#     ghz_qc_dc.h(j+1)
#     ghz_qc_dc.cx(j+1, j)

# ghz_qc_dc.barrier(label = "patch_bell")

# for j in range(1, N-1, 2):
#     ghz_qc_dc.cx(j, j+1)
#     ghz_qc_dc.measure(j+1, cr_syn[(j-1)//2])

# ghz_qc_dc.barrier(label = "stitching")

# for j in range(2, N-1, 2):
#     ghz_qc_dc.cx(j+1, j)

# ghz_qc_dc.barrier(label = "post_syn_meas")

# for j in range(2,N-1, 2):
#     m_ind = (j-1)//2

#     if m_ind == 0:
#         parity = expr.lift(cr_syn[m_ind])
#     else:
#         parity = expr.bit_xor(
#             cr_syn[m_ind], parity
#         )
    
#     with ghz_qc_dc.if_test(parity):
#         ghz_qc_dc.x(qr[j+1])
#         if j+2<N:
#             ghz_qc_dc.x(qr[j+2])

# ghz_qc_dc.barrier(label = "post_if_else")

# ghz_qc_dc.measure(qr, cr)
# ghz_qc_dc.draw("mpl", fold=-1)

In [7]:
# qc = [ghz_qc_ugc, ghz_qc_ugs, ghz_qc_si, ghz_qc_dc]

### Measurement Operator

The general dichotomic measurement operator on the Bloch sphere is

$$
A(\theta,\phi) \;=\; \sin\theta \cos\phi \, X
\;+\; \sin\theta \sin\phi \, Y
\;+\; \cos\theta \, Z,
$$

In [8]:
# angles = {'a1':{"theta": 1.5708, "phi": 0.396978},
#           'b1':{"theta": 1.5708, "phi": 0.396531},
#           'c1':{"theta": 1.5708, "phi": 1.00564},
#           'd1':{"theta": 1.5708, "phi": 0.460767},
#           'e1':{"theta": 1.5708, "phi": 1.00519},
#           'f1':{"theta": 1.5708, "phi": 1.00564},
#           'g1':{"theta": 1.5708, "phi": 1.00616},
#           'h1':{"theta": 1.5708, "phi": 1.00628},
#           'a2':{"theta": 1.5708, "phi": 1.18217},
#           'b2':{"theta": 1.5708, "phi": 1.18214},
#           'c2':{"theta": 1.5708, "phi": 0.691424},
#           'd2':{"theta": 1.5708, "phi": 2.03156},
#           'e2':{"theta": 1.5708, "phi": 0.691841},
#           'f2':{"theta": 1.5708, "phi": 0.691328},
#           'g2':{"theta": 1.5708, "phi": 0.691696},
#           'h2':{"theta": 1.5708, "phi": 0.691826}}

In [9]:
# X = Pauli('X')
# Y = Pauli('Y')
# Z = Pauli('Z')

# observables = {
#     'a1b1c1d1e1f1g1h1': {
#         "SparsePauliOp": {"Pauli_List": [], "Pauli_Coeff": []}, 'coeff': 1
#     },

#     'a2b2c1d2e1f1g1h1': {
#         "SparsePauliOp": {"Pauli_List": [], "Pauli_Coeff": []}, "coeff": -1
#     },

#     "a2b2c2d1e2f2g2h2": {
#         "SparsePauliOp": {"Pauli_List": [], "Pauli_Coeff": []}, "coeff": 1
#     },

#     "a1b1c2d2e2f2g2h2": {
#         "SparsePauliOp": {"Pauli_List": [], "Pauli_Coeff": []}, "coeff": 1
#     }    
# }

# for key in observables.keys():
#     pauli_op = None
#     for i in range(0,16, 2):
#         meas_key = key[i:i+2]
#         theta = angles[meas_key]["theta"]
#         phi = angles[meas_key]["phi"]
        
#         if pauli_op is None:
#             pauli_op = SparsePauliOp(['X','Y','Z'], [np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)])
#         else:
#             pauli_op = pauli_op.tensor(SparsePauliOp(['X','Y','Z'], [np.sin(theta)*np.cos(phi), np.sin(theta)*np.sin(phi), np.cos(theta)]))        
#     observables[key]["SparsePauliOp"]["Pauli_List"] = pauli_op.paulis.to_labels()
#     observables[key]["SparsePauliOp"]["Pauli_Coeff"] = pauli_op.coeffs

In [10]:
# state_0 = np.array([1,0])
# state_1 = np.array([0,1])
# kron_state_0 = state_0
# kron_state_1 = state_1
# for _ in range(7):
#     kron_state_0 = np.kron(kron_state_0, state_0)
#     kron_state_1 = np.kron(kron_state_1, state_1)
# ghz_state = (1/np.sqrt(2)) * (kron_state_0 + kron_state_1)

# unique_pauli_strings = OrderedSet()

# for term in observables.values():
#     labels = term["SparsePauliOp"]["Pauli_List"]
#     unique_pauli_strings.update(labels)

# non_zero_pauli_op = []

# for string in unique_pauli_strings:
#     obs = SparsePauliOp(string).to_matrix()
#     exp_value = ghz_state.conj().T @ obs @ ghz_state

#     if exp_value != 0:
#         non_zero_pauli_op.append(string)

In [11]:
# metadata = {
#     "angles" : angles,
#     "unique_pauli_strings" : unique_pauli_strings,
#     "non_zero_pauli_string" : non_zero_pauli_op
# }

# with open(r"data\metadata.pkl", "wb") as f: 
#     pickle.dump(metadata, f)

Only 129

In [12]:
# all_quantum_data_129 = {}

In [13]:
# # New Observables
# observables_new = {
#     'a1b1c1d1e1f1g1h1': {
#         "SparsePauliOp": {"Pauli_List": [], "Pauli_Coeff": []}, 'coeff': 1
#     },

#     'a2b2c1d2e1f1g1h1': {
#         "SparsePauliOp": {"Pauli_List": [], "Pauli_Coeff": []}, "coeff": -1
#     },

#     "a2b2c2d1e2f2g2h2": {
#         "SparsePauliOp": {"Pauli_List": [], "Pauli_Coeff": []}, "coeff": 1
#     },

#     "a1b1c2d2e2f2g2h2": {
#         "SparsePauliOp": {"Pauli_List": [], "Pauli_Coeff": []}, "coeff": 1
#     }    
# }

# for obs_key in observables.keys():
#     new_paulis = []
#     new_coeff = []

#     for pauli, coeff in zip(observables[obs_key]["SparsePauliOp"]["Pauli_List"], observables[obs_key]["SparsePauliOp"]["Pauli_Coeff"]):
#         if pauli in non_zero_pauli_op:
#             new_paulis.append(pauli)
#             new_coeff.append(coeff)
    
#     observables_new[obs_key]["SparsePauliOp"]["Pauli_List"] = new_paulis
#     observables_new[obs_key]["SparsePauliOp"]["Pauli_Coeff"] = new_coeff 

In [14]:
# # Simulator data (stores original circuits and observables)
# qc_list = {}
# for circuit in qc:
#     qc_data = {"circuit": None, "observables": copy.deepcopy(observables_new), "run_circuits": []}
#     qc_data["circuit"] = circuit

#     for obs_key in qc_data["observables"].keys():
#         circuit_no_meas = circuit.remove_final_measurements(inplace = False)
#         qregs = circuit.qregs[0]
#         cregs_meas = ClassicalRegister(8, name = "meas")
#         circuit_no_meas.add_register(cregs_meas)
#         circuit_no_meas.barrier(label = "post_if_else")
        
#         for i in range(8):
#             obs_str = obs_key[2*i:2*i+2]
#             theta = angles[obs_str]["theta"]
#             phi = angles[obs_str]["phi"]

#             circuit_no_meas.rz(-phi, qregs[i])
#             circuit_no_meas.ry(-theta, qregs[i])
#             circuit_no_meas.measure(qregs[i], cregs_meas[i])
        
#         qc_data["run_circuits"].append(circuit_no_meas)

#     qc_list[circuit.name] = qc_data

# all_quantum_data_129["simulator"] = qc_list

In [15]:
# with open(r"data\data_129.pkl", "wb") as f: 
#     pickle.dump(all_quantum_data_129, f)

Safety Data

In [16]:
# safety_data = {
#     "runtime":{}, #EstimatorOp : {circuit_name: time}
#     "services" : ["Reserach_IITJ", "Research_IITJ_hotmail", "Tomis_1"]
# }
# with open(r"data\safety_data.pkl", "wb") as f: 
#     pickle.dump(safety_data, f)

Circuits ran

In [17]:
# run_completed_circuits = {} # mitigation: backend: [circuit set]
# with open(r"data\run_completed_circuits.pkl", "wb") as f: 
#     pickle.dump(run_completed_circuits, f)

# Required Functions and Data Loading

In [18]:
def exp_from_counts(counts: dict):
    total_shots = 0
    exp_value = 0
    for res_str, count in counts.items():
        total_shots += count
        exp_value += ((-1)**res_str.count('1')) * count
    return exp_value/total_shots

In [19]:
def append_services(service_name):
    with open(r"data\safety_data.pkl", "rb") as f: 
        safety_data = pickle.load(f)
    
    safety_data["services"].append(service_name)

    with open(r"data\safety_data.pkl", "wb") as f: 
        pickle.dump(safety_data, f)

In [20]:
# QiskitRuntimeService.save_account(
#     channel = "ibm_quantum_platform",
#     token = "",
#     instance="",
#     name = "",
#     overwrite=True
# )
# append_services("")

In [21]:
def safety_check(circuit_name, estimator_op, backend_name):
    with open(r"data\safety_data.pkl", "rb") as f:
        safety_data = pickle.load(f)

    req_time = 0
    available_service = None

    if estimator_op not in safety_data["runtime"].keys():
        safety_data["runtime"][estimator_op] = {circuit_name: 1}
        req_time = 1
    
    else:
        if circuit_name not in safety_data["runtime"][estimator_op].keys():
            safety_data["runtime"][estimator_op][circuit_name] = 1
            req_time = 1
        
        else:
            req_time = safety_data["runtime"][estimator_op][circuit_name]

    for service_name in safety_data["services"]:
        service = QiskitRuntimeService(name = service_name)
        if service.usage()["usage_remaining_seconds"] >= req_time:
            available_service = service_name
            break

    with open(r"data\safety_data.pkl", "wb") as f: 
        pickle.dump(safety_data, f)
        
    if available_service == None:
        return None
    else:
        return available_service

In [22]:
def update_run_completion(reset = False, mitigation_tech = None, backend_name = None, circuit_name = None, job = None):
    with open(r"data\run_completed_circuits.pkl", "rb") as f:
        run_completed_circuits = pickle.load(f)
    with open(r"data\safety_data.pkl", "rb") as f:
        safety_data = pickle.load(f)

    if reset:
        for mitigation in run_completed_circuits.keys():
            for backend in run_completed_circuits[mitigation]:
                run_completed_circuits[mitigation][backend] = set()
    
    else: 
        if mitigation_tech not in run_completed_circuits.keys():
            run_completed_circuits[mitigation_tech] = {}
            run_completed_circuits[mitigation_tech][backend_name] = set()
            run_completed_circuits[mitigation_tech][backend_name].add(circuit_name)
        
        else:
            if backend_name not in run_completed_circuits[mitigation_tech]:
                run_completed_circuits[mitigation_tech][backend_name] = set()
                run_completed_circuits[mitigation_tech][backend_name].add(circuit_name)
            
            else:
                run_completed_circuits[mitigation_tech][backend_name].add(circuit_name)

        if mitigation_tech not in safety_data["runtime"].keys():
            safety_data["runtime"][mitigation_tech] = {circuit_name: 1}
        
        else:
            if circuit_name not in safety_data["runtime"][mitigation_tech].keys():
                safety_data["runtime"][mitigation_tech][circuit_name] = 1
            
            else:
                safety_data["runtime"][mitigation_tech][circuit_name] = 1
                
    with open(r"data\run_completed_circuits.pkl", "wb") as f: 
        pickle.dump(run_completed_circuits, f)
    with open(r"data\safety_data.pkl", "wb") as f: 
        pickle.dump(safety_data, f)

In [23]:
def get_run_completion(mitigation_tech, backend_name, circuit_name):
    with open(r"data\run_completed_circuits.pkl", "rb") as f:
        run_completed_circuits = pickle.load(f)

    if mitigation_tech not in run_completed_circuits.keys():
        return True
    
    else:
        if backend_name not in run_completed_circuits[mitigation_tech]:
            return True
        else:
            if circuit_name in run_completed_circuits[mitigation_tech][backend_name]:
                return False
            else:
                return True

In [24]:
with open(r"data\data_6561.pkl", "rb") as f:
    all_quantum_data_6561 = pickle.load(f)
with open(r"data\data_129.pkl", "rb") as f:
    all_quantum_data_129 = pickle.load(f)
with open(r"data\metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

<a id="Data Structure Definition"></a>
<div style="background-color: #d4edda; color: #155724; padding: 15px; border-radius: 5px; font-family: monospace;">
    
<b>Data Structure Definition</b> 

<pre>
all_quantum_data_129 (dict)
└── simulator/backend_name (dict)
    └── circuit_name (dict)
        ├── circuit : QuantumCircuit object
        └── observables (dict)
        |   └── obs_name (dict)
        |       ├── SparsePauliOp : dict
        |       |   ├── Pauli_List: list
        |       |   └── Pauli_Coeff: list
        |       └── coeff : numeric coefficient
        └── run_data (dict)
            ├── circuit_list: list(isa_qc)
            └── unique_observables: list(str)

metadata (dict)
├── angles (dict)
│   └── 1qubit obs_name (dict)
│       ├── theta : float
│       └── phi : float
├── unique_pauli_strings : list of strings
└── non_zero_pauli_string : list of strings
</pre>

</div>


# Theoritical Values

In [25]:
state_0 = np.array([1,0])
state_1 = np.array([0,1])

kron_state_0 = state_0
kron_state_1 = state_1
for _ in range(7):
    kron_state_0 = np.kron(kron_state_0, state_0)
    kron_state_1 = np.kron(kron_state_1, state_1)

ghz_state = (1/np.sqrt(2)) * (kron_state_0 + kron_state_1)

In [26]:
print("Expectation values for the inequality terms are:")
S = 0
for key in all_quantum_data_6561["simulator"]["GHZ_UGC"]["observables"].keys():
    matrix = SparsePauliOp(all_quantum_data_6561["simulator"]["GHZ_UGC"]["observables"][key]["SparsePauliOp"]["Pauli_List"], all_quantum_data_6561["simulator"]["GHZ_UGC"]["observables"][key]["SparsePauliOp"]["Pauli_Coeff"]).to_matrix()
    exp_value = ghz_state.conj().T @ matrix @ ghz_state
    S += all_quantum_data_6561["simulator"]["GHZ_UGC"]["observables"][key]["coeff"] * exp_value

    print(f"{key}: {exp_value}")

print(f"\nClassical Bound: 2")
print(f"Quantum Bound: {np.real(S.round())}")

Expectation values for the inequality terms are:
a1b1c1d1e1f1g1h1: (0.9999999999457889+0j)
a2b2c1d2e1f1g1h1: (-0.9999999999439504+0j)
a2b2c2d1e2f2g2h2: (0.9999999999236325+0j)
a1b1c2d2e2f2g2h2: (0.9999999999451756+0j)

Classical Bound: 2
Quantum Bound: 4.0


In [27]:
print(f"Unique 8-qubit Pauli strings: {metadata["unique_pauli_strings"]}")
print(f"Total unique Pauli strings: {len(metadata["unique_pauli_strings"])}")
print(f"Unique 8-qubit Non-Zero Pauli strings: {metadata['non_zero_pauli_string']}")
print(f"Total unique Pauli strings: {len(metadata['non_zero_pauli_string'])}")
print(f"Non-compact Pauli required: {3**8}")

Unique 8-qubit Pauli strings: OrderedSet(['XXXXXXXX', 'XXXXXXXY', 'XXXXXXXZ', 'XXXXXXYX', 'XXXXXXYY', 'XXXXXXYZ', 'XXXXXXZX', 'XXXXXXZY', 'XXXXXXZZ', 'XXXXXYXX', 'XXXXXYXY', 'XXXXXYXZ', 'XXXXXYYX', 'XXXXXYYY', 'XXXXXYYZ', 'XXXXXYZX', 'XXXXXYZY', 'XXXXXYZZ', 'XXXXXZXX', 'XXXXXZXY', 'XXXXXZXZ', 'XXXXXZYX', 'XXXXXZYY', 'XXXXXZYZ', 'XXXXXZZX', 'XXXXXZZY', 'XXXXXZZZ', 'XXXXYXXX', 'XXXXYXXY', 'XXXXYXXZ', 'XXXXYXYX', 'XXXXYXYY', 'XXXXYXYZ', 'XXXXYXZX', 'XXXXYXZY', 'XXXXYXZZ', 'XXXXYYXX', 'XXXXYYXY', 'XXXXYYXZ', 'XXXXYYYX', 'XXXXYYYY', 'XXXXYYYZ', 'XXXXYYZX', 'XXXXYYZY', 'XXXXYYZZ', 'XXXXYZXX', 'XXXXYZXY', 'XXXXYZXZ', 'XXXXYZYX', 'XXXXYZYY', 'XXXXYZYZ', 'XXXXYZZX', 'XXXXYZZY', 'XXXXYZZZ', 'XXXXZXXX', 'XXXXZXXY', 'XXXXZXXZ', 'XXXXZXYX', 'XXXXZXYY', 'XXXXZXYZ', 'XXXXZXZX', 'XXXXZXZY', 'XXXXZXZZ', 'XXXXZYXX', 'XXXXZYXY', 'XXXXZYXZ', 'XXXXZYYX', 'XXXXZYYY', 'XXXXZYYZ', 'XXXXZYZX', 'XXXXZYZY', 'XXXXZYZZ', 'XXXXZZXX', 'XXXXZZXY', 'XXXXZZXZ', 'XXXXZZYX', 'XXXXZZYY', 'XXXXZZYZ', 'XXXXZZZX', 'XXXXZZZY'

# Simulator

In [28]:
from qiskit_aer.primitives import SamplerV2 as AerSampler

129

In [29]:
simulator = AerSampler()

results = {}

for circuit_name in all_quantum_data_129["simulator"].keys():
    results[circuit_name] = {} 
    pubs = [(circuit) for circuit in all_quantum_data_129["simulator"][circuit_name]["run_circuits"]]   
    run_res = simulator.run(pubs, shots=10000).result()

    S = 0
    for res, obs_key in zip(run_res, all_quantum_data_129["simulator"][circuit_name]["observables"].keys()):
        counts = res.data.meas.get_counts()
        exp_value = exp_from_counts(counts)
        S += all_quantum_data_129["simulator"][circuit_name]["observables"][obs_key]["coeff"] * exp_value
        results[circuit_name][obs_key] = exp_value
        
    results[circuit_name]["S"] = S
    print(f"{circuit_name} COMPLETED!")  

GHZ_UGC COMPLETED!
GHZ_UGS COMPLETED!
GHZ_SI COMPLETED!
GHZ_DC COMPLETED!


In [30]:
df = pd.DataFrame(data = results)
display(df)

,GHZ_UGC,GHZ_UGS,GHZ_SI,GHZ_DC
a1b1c1d1e1f1g1h1,1.0,1.0,1.0,1.0
a2b2c1d2e1f1g1h1,-1.0,-1.0,-1.0,-1.0
a2b2c2d1e2f2g2h2,1.0,1.0,1.0,1.0
a1b1c2d2e2f2g2h2,1.0,1.0,1.0,1.0
S,4.0,4.0,4.0,4.0


In [31]:
df.to_csv(r"results\Results_simulator_129.csv", index=False)

6561

In [32]:
# simulator = AerSampler()

# results = {}

# for circuit_name in all_quantum_data_6561["simulator"].keys():
#     results[circuit_name] = {} 
#     pubs = [(circuit) for circuit in all_quantum_data_6561["simulator"][circuit_name]["run_data"]["circuit_list"]]   
#     run_res = simulator.run(pubs, shots=10000).result()

#     res_obj={}
#     for res, obs_name in zip(run_res, all_quantum_data_6561["simulator"][circuit_name]["run_data"]["unique_observables"]):
#         counts = res.data.meas.get_counts()
#         exp_value = exp_from_counts(counts)
#         res_obj[obs_name] = exp_value

#     S = 0
#     for obs_key in all_quantum_data_6561["simulator"][circuit_name]["observables"].keys():
#         exp_value = 0
#         for pauli, coeff in zip(all_quantum_data_6561["simulator"][circuit_name]["observables"][obs_key]["SparsePauliOp"]["Pauli_List"], 
#                                 all_quantum_data_6561["simulator"][circuit_name]["observables"][obs_key]["SparsePauliOp"]["Pauli_Coeff"]):
#             exp_value += coeff * res_obj[pauli]
#         S += all_quantum_data_6561["simulator"][circuit_name]["observables"][obs_key]["coeff"] * exp_value

#         results[circuit_name][obs_key] = exp_value
#     results[circuit_name]["S"] = S
#     print(f"{circuit_name} COMPLETED!")  

In [33]:
# df = pd.DataFrame(data = results)
# display(df)

In [34]:
# df.to_csv(r"results\Results_simulator_6561.csv")

# 4. Hardware

In [35]:
sampler_options_list = {
    "no_mitigation" : SamplerOptions(
        default_shots = 10000,
        dynamical_decoupling = {
            "enable" : False,
            "scheduling_method" : 'alap'
        },
        twirling = {
            "enable_gates" : False,
            "enable_measure" : False, 
        }         
    ),
    "DD_only" : SamplerOptions(
        default_shots = 10000,
        dynamical_decoupling = {
            "enable" : True,
            "scheduling_method" : 'alap'
        },
        twirling = {
            "enable_gates" : False,
            "enable_measure" : False,
        }
    ),
    "measure_only" : SamplerOptions(
        default_shots = 10000,
        dynamical_decoupling = {
            "enable" : False,
            "scheduling_method" : 'alap'
        },
        twirling = {
            "enable_gates" : False,
            "enable_measure" : True,
        }
    ),
    "gates_only" : SamplerOptions(
        default_shots = 10000,
        dynamical_decoupling = {
            "enable" : False,
            "scheduling_method" : 'alap'
        },
        twirling = {
            "enable_gates" : True,
            "enable_measure" : False,
        }
    ),
    "dd_measure" : SamplerOptions(
        default_shots = 10000,
        dynamical_decoupling = {
            "enable" : True,
            "scheduling_method" : 'alap'
        },
        twirling = {
            "enable_gates" : False,
            "enable_measure" : True,
        }
    ),
    "dd_gates" : SamplerOptions(
        default_shots = 10000,
        dynamical_decoupling = {
            "enable" : True,
            "scheduling_method" : 'alap'
        },
        twirling = {
            "enable_gates" : True,
            "enable_measure" : False,
        }
    ),
    "twirling" : SamplerOptions(
        default_shots = 10000,
        dynamical_decoupling = {
            "enable" : False,
            "scheduling_method" : 'alap'
        },
        twirling = {
            "enable_gates" : True,
            "enable_measure" : True,
        }
    ),
    "full_mitigation" : SamplerOptions(
        default_shots = 10000,
        dynamical_decoupling = {
            "enable" : True,
            "scheduling_method" : 'alap'
        },
        twirling = {
            "enable_gates" : True,
            "enable_measure" : True,
        }
    )
}

In [36]:
from qiskit.transpiler.passes import ALAPScheduleAnalysis, ASAPScheduleAnalysis, PadDynamicalDecoupling
from qiskit.circuit.library import XGate
from qiskit.circuit import Delay, QuantumCircuit
from qiskit.transpiler import PassManager
from qiskit.converters import circuit_to_dag
from collections import defaultdict
import warnings
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic import build, Twirl

warnings.filterwarnings("ignore", message=".*CircuitInstruction.*")
warnings.filterwarnings("ignore", message=".*Instruction.condition.*")

def split_qc(full_circuit: QuantumCircuit):
    """Split circuit into pre, mid, and post sections based on barriers."""
    # 1. Create empty circuits with the same registers as the input
    qc_pre = QuantumCircuit(*full_circuit.qregs, *full_circuit.cregs, name="Pre_If_else_Physical")
    qc_mid = QuantumCircuit(*full_circuit.qregs, *full_circuit.cregs, name="Mid_If_else_Physical")
    qc_post = QuantumCircuit(*full_circuit.qregs, *full_circuit.cregs, name="Post_If_else_Physical")

    # We start filling the first part
    current_circuit = qc_pre
    
    # 2. Iterate through every instruction in the original circuit
    for instruction in full_circuit.data:
        op = instruction.operation
        op_name = op.name

        # TRIGGER 1: Switch to Part 2 upon the first Measurement
        if current_circuit == qc_pre and op_name == 'barrier' and op.label == "post_syn_meas":
            current_circuit = qc_mid
            
        # TRIGGER 2: Switch to Part 3 upon the first Reset (|0>)
        elif current_circuit == qc_mid and op_name == 'barrier' and op.label == "post_if_else":
            current_circuit = qc_post
            
        # Append the instruction to the currently active circuit
        current_circuit.append(instruction.operation, instruction.qubits, instruction.clbits)

    return qc_pre, qc_mid, qc_post


def get_active_qubits(circuit: QuantumCircuit):
    """Get list of active qubits in the circuit."""
    active_qubits = set()

    for instruction in circuit.data:
        # We usually ignore barriers as they are just directives for the transpiler
        # and don't constitute "physical" activity on the qubit.
        if instruction.operation.name != 'barrier':
            for qubit in instruction.qubits:
                # find_bit returns the index of the qubit in the circuit
                index = circuit.find_bit(qubit).index
                active_qubits.add(index)

    return sorted(list(active_qubits))


def apply_paddd_asap(block: QuantumCircuit, backend):
    """Apply DD using ASAP scheduling."""
    DD_SEQUENCE = [XGate(), XGate()]

    pm = PassManager([
        ASAPScheduleAnalysis(backend.instruction_durations),
        PadDynamicalDecoupling(
            durations=backend.instruction_durations,
            dd_sequence=DD_SEQUENCE,
        ),
    ])

    return pm.run(block)


def apply_paddd_alap(block: QuantumCircuit, backend):
    """Apply DD using ALAP scheduling."""
    DD_SEQUENCE = [XGate(), XGate()]

    pm = PassManager([
        ALAPScheduleAnalysis(backend.instruction_durations),
        PadDynamicalDecoupling(
            durations=backend.instruction_durations,
            dd_sequence=DD_SEQUENCE,
        ),
    ])

    return pm.run(block)


def add_stretch_dd(qc, q, name):
    """Add stretch DD sequence to a qubit."""
    s = qc.add_stretch(name)
    qc.delay(s, q)
    qc.x(q)
    qc.delay(s, q)
    qc.x(q)
    qc.delay(s, q)
    return qc


def inject_dd_during_measure(qc_mid: QuantumCircuit, backend, active_qubits: list, num_dd=1):
    """Inject DD sequences during measurement period for idle qubits."""
    qubits_with_if_else = []
    qubits_without_if_else = []

    for idx, instruction in enumerate(qc_mid.data):
        operation = instruction.operation
        qubits = instruction.qubits
        
        # Check if this is an if_else operation
        if operation.name == 'if_else':
            # Get the qubit indices involved
            qubit_indices = [qc_mid.find_bit(q).index for q in qubits]
            
            # Check if any active qubits are involved
            if any(q in active_qubits for q in qubit_indices):
                # Add qubits involved in if_else to the list
                for q in qubit_indices:
                    if q in active_qubits and q not in qubits_with_if_else:
                        qubits_with_if_else.append(q)
    
    # Determine which active qubits don't have if_else
    qubits_without_if_else = [q for q in active_qubits if q not in qubits_with_if_else]

    qc_mid_new = QuantumCircuit(*qc_mid.qregs, *qc_mid.cregs, name="Mid_If_else_new")

    # Find the first barrier in the original circuit
    first_barrier_found = False

    # Iterate through original circuit instructions
    for idx, instruction in enumerate(qc_mid.data):
        operation = instruction.operation
        qubits = instruction.qubits
        clbits = instruction.clbits
        
        # Copy the instruction to new circuit
        qc_mid_new.append(operation, qubits, clbits)
        
        # After the first barrier, add DD to qubits WITHOUT if_else
        if operation.name == 'barrier' and not first_barrier_found:
            first_barrier_found = True
            
            # Add DD to qubits WITHOUT if_else at the START (num_dd sequences)
            for q in qubits_without_if_else:
                for dd_idx in range(num_dd):
                    add_stretch_dd(qc_mid_new, q, f"s_without_{q}_dd{dd_idx}")

    return qc_mid_new


def apply_dd_dc(qc: QuantumCircuit, backend, num_dd=1):
    """
    Apply DD to a dynamic circuit without twirling.
    
    This is the original function - applies DD to all three sections.
    """
    before, mid, after = split_qc(qc)
    data_qubits = get_active_qubits(qc)

    if before:
        before = apply_paddd_alap(before, backend)

    if mid:
        mid = inject_dd_during_measure(mid, backend, data_qubits, num_dd)

    if after:
        after = apply_paddd_asap(after, backend)

    out = QuantumCircuit(*qc.qregs, *qc.cregs, name=qc.name + "_DD")
    out.compose(before, inplace=True)
    out.compose(mid, inplace=True)
    out.compose(after, inplace=True)

    return out

In [37]:
def insert_mid_at_barrier(template, qc_mid, barrier_label):
    """Split circuit into pre, mid, and post sections based on barriers."""
    # 1. Create empty circuits with the same registers as the input
    qc_pre = QuantumCircuit(*template.qregs, *template.cregs, name="Pre_If_else_Physical")
    qc_post = QuantumCircuit(*template.qregs, *template.cregs, name="Post_If_else_Physical")

    # We start filling the first part
    current_circuit = qc_pre
    
    # 2. Iterate through every instruction in the original circuit
    for instruction in template.data:
        op = instruction.operation
        op_name = op.name

        if current_circuit == qc_pre and op_name == 'barrier' and op.label == barrier_label:
            current_circuit = qc_post
            continue
            
        # Append the instruction to the currently active circuit
        current_circuit.append(instruction.operation, instruction.qubits, instruction.clbits)

    qc_pre.compose(qc_mid, inplace=True)
    qc_pre.compose(qc_post, inplace=True)

    return qc_pre

def apply_full_mitigation_dc(qc: QuantumCircuit, backend, preset_pm, num_dd=1, num_randomizations=100):
    before, mid, after = split_qc(qc)

    if before:  
        boxing_pm = generate_boxing_pass_manager(
            enable_gates=True,
            enable_measures=False,
            remove_barriers="never"
        )

        before = boxing_pm.run(before)
    
    if after:  
        boxing_pm = generate_boxing_pass_manager(
            enable_gates=True,
            enable_measures=True,
            remove_barriers="never"
        )

        after = boxing_pm.run(after)
        
    # Step 1: Combine before and after circuits with a named barrier
    combined_qc = before.copy()
    combined_qc.barrier(label='cut_here')
    combined_qc.compose(after, inplace=True)

    # Step 3: Build samplex with twirling
    template, samplex = build(combined_qc)
    outputs = samplex.sample({}, num_randomizations=num_randomizations)

    template = insert_mid_at_barrier(template, mid, "cut_here")
    template = preset_pm.run(template)
    template = apply_dd_dc(template, backend, num_dd)

    return template, outputs

### Test Mitigation

In [38]:
# service = QiskitRuntimeService(name="vedant")

In [39]:
# backend = service.backend("ibm_marrakesh")

In [40]:
# pm_lv3 = generate_preset_pass_manager(optimization_level=3, backend=backend, scheduling_method=None, seed_transpiler=42)

In [41]:
# results_mitigation = {}
# ghz_qc = all_quantum_data_129["simulator"]["GHZ_DC"]["run_circuits"][0]

No Mitigation

In [42]:
# sampler_dc = SamplerV2(mode = backend, options = sampler_options_list["no_mitigation"])
# pubs_dc = [(ghz_qc)]
# job_dc = sampler_dc.run(pubs_dc, shots=10000)
# print(f"Job_DC = {job_dc}")

In [43]:
# results_mitigation["no_mitigation"] = {}
# results_mitigation["no_mitigation"]["GHZ_DC"] = exp_from_counts(job_dc.result()[0].data.meas.get_counts())
# print(f"{results_mitigation["no_mitigation"]["GHZ_DC"]} took {job_dc.metrics()["usage"]["quantum_seconds"]}s")

DD_only

In [44]:
# sampler_dc = SamplerV2(mode = backend, options = sampler_options_list["no_mitigation"])
# pubs_dc = [apply_dd_dc(ghz_qc, backend, 0)]
# job_dc = sampler_dc.run(pubs_dc, shots=10000)
# print(f"Job_DC = {job_dc}")

Measure_only

In [45]:
# sampler_measure_only = SamplerV2(mode = backend, options = sampler_options_list["measure_only"])
# pubs_measure_only = [(ghz_qc)]
# job_measure_only = sampler_measure_only.run(pubs_measure_only, shots=10000)
# print(f"Job measure_only = {job_measure_only.job_id()}")

In [46]:
# results_mitigation["measure_only"] = {}
# results_mitigation["measure_only"]["GHZ_DC"] = exp_from_counts(job_measure_only.result()[0].data.meas.get_counts())
# print(f"{results_mitigation["measure_only"]["GHZ_DC"]} took {job_measure_only.metrics()["usage"]["quantum_seconds"]}s")

Gates_only

In [47]:
# sampler_gates_only = SamplerV2(mode = backend, options = sampler_options_list["gates_only"])
# pubs_gates_only = [(ghz_qc)]
# job_gates_only = sampler_gates_only.run(pubs_gates_only, shots=10000)
# print(f"Job gates_only = {job_gates_only.job_id()}")

In [48]:
# results_mitigation["gates_only"] = {}
# results_mitigation["gates_only"]["GHZ_DC"] = exp_from_counts(job_gates_only.result()[0].data.meas.get_counts())
# print(f"{results_mitigation["gates_only"]["GHZ_DC"]} took {job_gates_only.metrics()["usage"]["quantum_seconds"]}s")

dd+measure

In [49]:
# sampler_dc = SamplerV2(mode = backend, options = sampler_options_list["measure_only"])
# pubs_dc = [(apply_dd_dc(ghz_qc, backend, 0))]
# job_dc = sampler_dc.run(pubs_dc, shots=10000)
# print(f"Job_DC = {job_dc}")

In [50]:
# results_mitigation["dd_measure"] = {}
# results_mitigation["dd_measure"]["GHZ_DC"] = exp_from_counts(job_dc.result()[0].data.meas.get_counts())
# print(f"{results_mitigation["dd_measure"]["GHZ_DC"]} took {job_dc.metrics()["usage"]["quantum_seconds"]}s")

dd+gates

In [51]:
# sampler_dc = SamplerV2(mode = backend, options = sampler_options_list["gates_only"])
# pubs_dc = [(apply_dd_dc(ghz_qc, backend, 0))]
# job_dc = sampler_dc.run(pubs_dc, shots=10000)
# print(f"Job_DC = {job_dc}")

In [52]:
# results_mitigation["dd_gates"] = {}
# results_mitigation["dd_gates"]["GHZ_DC"] = exp_from_counts(job_dc.result()[0].data.meas.get_counts())
# print(f"{results_mitigation["dd_gates"]["GHZ_DC"]} took {job_dc.metrics()["usage"]["quantum_seconds"]}s")

Twirling

In [53]:
# sampler_twirling = SamplerV2(mode = backend, options = sampler_options_list["twirling"])
# pubs_twirling = [(ghz_qc_dc)]
# job_twirling = sampler_twirling.run(pubs_twirling, shots=10000)
# print(f"Job twirling = {job_twirling.job_id()}")

In [54]:
# results_mitigation["twirling"] = {}
# results_mitigation["twirling"]["GHZ_DC"] = exp_from_counts(service.job("d5n6qkk8d8hc73chd8ig").result()[0].data.meas.get_counts())

full_mitigation

In [55]:
# #DC
# from qiskit.primitives.containers import BitArray

# sampler_dc = SamplerV2(mode = backend, options = sampler_options_list["no_mitigation"])
# template, outputs = apply_full_mitigation_dc(ghz_qc, backend, pm_lv3, 1, 32)
# job_dc = sampler_dc.run([(template, outputs["parameter_values"])], shots=10000)
# print(f"Job_DC = {job_dc}")

In [56]:
# with open(r"data/output.pkl", "wb") as f:
#     pickle.dump(outputs, f)

In [57]:
# with open(r"data/output.pkl", "rb") as f:
#     outputs = pickle.load(f)

# print(outputs)

In [58]:
# outputs["measurement_flips.cr_syn"]

d5tnlfohusoc73er9pmg

In [59]:
# job_dc = service.job("d5u4j2ohusoc73ero69g")
# # job_dc.status()

In [60]:
# meas_data = job_dc.result()[0].data["meas"]
# meas_data ^= BitArray.from_bool_array(outputs["measurement_flips.meas"], "little")
# print(meas_data)

In [61]:
# total_exp = 0

# for i in range(32):
#     counts = meas_data[i].get_counts()
#     exp_val = exp_from_counts(counts)
#     total_exp+=exp_val
#     print(f"Ckt_{i}: {exp_val}")

# print(f"Final exp: {total_exp/32}")

m3

In [62]:
# import mthree as m3
# from qiskit.result import marginal_counts

In [63]:
# result = copy.deepcopy(job_dc.result()[0])
# joined_bits_original = result.join_data()

# # Create a new list with XOR-corrected BitArrays
# joined_bits = []

# for i in range(32):
#     # Get the flip pattern
#     flip_pattern = outputs["measurement_flips.meas"][i]
#     flip_array = np.array(flip_pattern, dtype=bool)
    
#     # Calculate padding size
#     total_bits = joined_bits_original[i].num_bits
#     meas_bits = flip_array.shape[1]
#     cr_syn_bits = total_bits - meas_bits
    
#     # Create padded flip
#     num_shots = flip_array.shape[0]
#     padding = np.zeros((num_shots, cr_syn_bits), dtype=bool)
#     padded_flip = np.concatenate([padding, flip_array], axis=1)
    
#     # Apply XOR and append to new list
#     flip_bitarray = BitArray.from_bool_array(padded_flip, "little")
#     corrected_bitarray = joined_bits_original[i] ^ flip_bitarray
#     joined_bits.append(corrected_bitarray)

In [64]:
# counts_dc = [
#     joined_bits[i].get_counts()
#     for i in range(32)
# ]
# mappings_dc = m3.utils.final_measurement_mapping(template)
# mit = m3.M3Mitigation(backend)
# mit.cals_from_system(mappings_dc)
# quasis = [
#     mit.apply_correction(counts, mappings_dc)
#     for counts in counts_dc
# ]
# corrected_counts_dc = [
#     {k: round(v * 10000) for k, v in quasi.items()}
#     for quasi in quasis
# ]

# final_counts = [
#     marginal_counts(counts_dc[i], indices = list(range(3,11)))
#     for i in range(32)
# ]

# final_corrected_counts = [
#     marginal_counts(corrected_counts_dc[i], indices = list(range(3,11)))
#     for i in range(32)
# ]

In [65]:
# total_exp = 0

# for i in range(32):
#     exp_val = exp_from_counts(final_counts[i])
#     total_exp+=exp_val
#     print(f"Ckt_{i}: {exp_val}")

# print(f"Final exp: {total_exp/32}")

In [66]:
# total_exp = 0

# for i in range(32):
#     exp_val = exp_from_counts(final_corrected_counts[i])
#     total_exp+=exp_val
#     print(f"Ckt_{i}: {exp_val}")

# print(f"Final exp: {total_exp/32}")

In [67]:
# counts_dc = service.job("d5u4j2ohusoc73ero69g").result()[0].join_data().get_counts()
# mappings_dc = m3.utils.final_measurement_mapping(ghz_qc)
# mit = m3.M3Mitigation(backend)
# mit.cals_from_system(mappings_dc)
# quasis = mit.apply_correction(counts_dc, mappings_dc)
# corrected_counts_dc = {key: round(value * 10000) for key, value in quasis.items()}

In [68]:
# final_counts = marginal_counts(corrected_counts_dc, indices = list(range(3,11)))
# results_mitigation["m3"] = {}
# results_mitigation["m3"]["GHZ_DC"] = exp_from_counts(final_counts)
# print(f"{results_mitigation["m3"]["GHZ_DC"]}")

In [69]:
# sampler_dc = SamplerV2(mode = backend, options = full_mitigation_dc)
# pubs_dc = [(apply_dd_dc(ghz_qc, backend, 1))]
# job_dc = sampler_dc.run(pubs_dc, shots=10000)
# print(f"Job_DC = {job_dc}")

In [70]:
# results_mitigation["full_mitigation"] = {}
# results_mitigation["full_mitigation"]["GHZ_DC"] = exp_from_counts(job_dc.result()[0].data.meas.get_counts())
# print(f"{results_mitigation["full_mitigation"]["GHZ_DC"]} took {job_dc.metrics()["usage"]["quantum_seconds"]}s")

Store Data

In [71]:
# df_mitigation = pd.DataFrame(results_mitigation)
# display(df_mitigation)

In [72]:
# df_mitigation.to_csv(r"results\mitigation_test_data_DC.csv")

### Actual Runs

In [73]:
run_data = pd.read_csv(r"results\Run_data.csv")
display(run_data)

,date,time,job_id,service_acc,circuit,mitigation,backend,S,a1b1c1d1e1f1g1h1,a2b2c1d2e1f1g1h1,a2b2c2d1e2f2g2h2,a1b1c2d2e2f2g2h2,Ideal S,Ideal_a1b1c1d1e1f1g1h1,Ideal_a2b2c1d2e1f1g1h1,Ideal_a2b2c2d1e2f2g2h2,Ideal_a1b1c2d2e2f2g2h2
0,19-01-2026,14:50:34,d5mvdmbh36vs73bilvc0,swapnil_charutarhealth,GHZ_UGC,no_mitigation,ibm_fez,2.983600,0.743200,-0.731200,0.748800,0.760400,4,1,-1,1,1
1,19-01-2026,14:51:22,d5mvdmbh36vs73bilvc0,swapnil_charutarhealth,GHZ_UGC,no_mitigation+m3,ibm_fez,3.380737,0.837984,-0.830932,0.848248,0.863573,4,1,-1,1,1
2,19-01-2026,17:33:47,d5n0bet9j2ac739lc5ug,swapnil_charutarhealth,GHZ_UGC,full_mitigation,ibm_marrakesh,2.980800,0.730200,-0.759000,0.740600,0.751000,4,1,-1,1,1
3,19-01-2026,17:57:52,d5n0bet9j2ac739lc5ug,swapnil_charutarhealth,GHZ_UGC,full_mitigation+m3,ibm_marrakesh,3.326079,0.807723,-0.844216,0.815455,0.858685,4,1,-1,1,1
4,19-01-2026,18:10:44,d5n0a1ph2mqc739bdl70,swapnil_charutarhealth,GHZ_DC,full_mitigation,ibm_fez,1.409400,0.345800,-0.355800,0.340800,0.367000,4,1,-1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,24-01-2026,16:13:35,d5qa4g8h0i0s73eq72b0,vedant,GHZ_UGS,full_mitigation+m3,ibm_marrakesh,2.746612,0.692569,-0.689476,0.683158,0.681409,4,1,-1,1,1
64,30-01-2026,22:51:26,d5udf74cqoec73dkr4eg,vedant,GHZ_DC,no_mitigation,ibm_marrakesh,1.277400,0.302000,-0.333200,0.322800,0.319400,4,1,-1,1,1
65,31-01-2026,10:40:49,d5udf74cqoec73dkr4eg,vedant,GHZ_DC,no_mitigation+m3,ibm_marrakesh,1.365761,0.321803,-0.357970,0.346319,0.339668,4,1,-1,1,1
66,31-01-2026,12:58:05,d5uqtspfodos73emompg,vedant,GHZ_DC,no_mitigation,ibm_fez,0.830800,0.202400,-0.214400,0.207200,0.206800,4,1,-1,1,1


In [74]:
def store_run_results(run_data, job_id, service, circuit, backend, mitigation, S, a1b1c1d1e1f1g1h1, a2b2c1d2e1f1g1h1, a2b2c2d1e2f2g2h2, a1b1c2d2e2f2g2h2):
    row = {
        "date": time.strftime("%d-%m-%Y"),
        "time": time.strftime("%H:%M:%S"),
        "job_id": job_id,
        "service_acc" : service,
        "circuit": circuit,
        "backend": backend,
        "mitigation": mitigation,
        "S": S,
        "a1b1c1d1e1f1g1h1": a1b1c1d1e1f1g1h1,
        "a2b2c1d2e1f1g1h1": a2b2c1d2e1f1g1h1,
        "a2b2c2d1e2f2g2h2": a2b2c2d1e2f2g2h2,
        "a1b1c2d2e2f2g2h2": a1b1c2d2e2f2g2h2,
        "Ideal S" : 4,
        "Ideal_a1b1c1d1e1f1g1h1": 1, 
        "Ideal_a2b2c1d2e1f1g1h1" : -1,
        "Ideal_a2b2c2d1e2f2g2h2": 1, 
        "Ideal_a1b1c2d2e2f2g2h2": 1
    }
    row_df = pd.DataFrame([row])
    run_data = pd.concat([run_data, row_df], ignore_index=True)
    return run_data

Only 129

In [75]:
backend_list = ['ibm_fez','ibm_marrakesh'] # 'ibm_torino' ,  
circuit_list = ["GHZ_DC"] #, "GHZ_UGC"

In [76]:
# update_run_completion(reset = True)

In [77]:
static_service = QiskitRuntimeService(name="Vishal")

In [78]:
sampler_options_list = {
    "no_mitigation" : SamplerOptions(
        default_shots = 10000,
        dynamical_decoupling = {
            "enable" : False,
            "scheduling_method" : 'alap'
        },
        twirling = {
            "enable_gates" : False,
            "enable_measure" : False, 
        }         
    ),
    "full_mitigation" : SamplerOptions(
        default_shots = 10000,
        dynamical_decoupling = {
            "enable" : True,
            "scheduling_method" : 'alap'
        },
        twirling = {
            "enable_gates" : True,
            "enable_measure" : True,
        }
    )
}

In [ ]:
# for mitigation_tech in sampler_options_list.keys():
#     for backend_name in backend_list:
#         for circuit_name in all_quantum_data_129["simulator"].keys():
#             if (get_run_completion(mitigation_tech, backend_name, circuit_name) == False) or (circuit_name not in circuit_list):
#                 print(f"{circuit_name} on {backend_name} with {mitigation_tech} skipped!")
#                 continue

#             available_service_name = safety_check(circuit_name, mitigation_tech, backend_name)
#             if available_service_name == None:
#                 raise(Exception("NO SERVICE AVAILABLE!!!"))
#             available_service = QiskitRuntimeService(name = available_service_name)
#             print(f"Available_serive : {next(k for k, v in static_service.saved_accounts().items() if v == static_service.active_account())}")
#             backend = available_service.backend(name = backend_name)
#             if backend == None:
#                 raise(Exception("No Backend Available!!!"))
                       
#             pm_lv3 = generate_preset_pass_manager(optimization_level=3, backend=backend, scheduling_method=None, seed_transpiler=42)
#             isa_qc_list = [pm_lv3.run(qc) for qc in all_quantum_data_129["simulator"][circuit_name]["run_circuits"]]

#             if circuit_name == "GHZ_DC" and mitigation_tech=="full_mitigation":
#                 continue
#             else:
#                 sampler = SamplerV2(mode = backend, options = sampler_options_list[mitigation_tech])
#                 pubs = [(isa_qc) for isa_qc in isa_qc_list]

#             print(f"Running {circuit_name} on {backend_name} with {mitigation_tech}")    
#             try:
#                 job = sampler.run(pubs, shots = 10000)  
#                 run_result = job.result()
#             except Exception as e:
#                 raise Exception("Job failed") from e
                
#             print(f"Done Running {circuit_name} on {backend_name}. Took {job.metrics()["usage"]["quantum_seconds"]}s")
#             update_run_completion(False, mitigation_tech, backend_name, circuit_name, job)

#             final_result = {'S': 0, "a1b1c1d1e1f1g1h1": 0, "a2b2c1d2e1f1g1h1": 0, "a2b2c2d1e2f2g2h2": 0, "a1b1c2d2e2f2g2h2": 0}

#             for res,obs_key in zip(run_result ,all_quantum_data_129["simulator"][circuit_name]["observables"].keys()):
#                 counts = res.data.meas.get_counts()
#                 exp_value = exp_from_counts(counts)
#                 final_result[obs_key] = exp_value
#                 final_result['S'] += all_quantum_data_129["simulator"][circuit_name]["observables"][obs_key]["coeff"] * exp_value

#             run_data = store_run_results(run_data, job.job_id(), available_service_name, circuit_name, backend_name, mitigation_tech, final_result["S"], final_result["a1b1c1d1e1f1g1h1"],
#                                         final_result["a2b2c1d2e1f1g1h1"], final_result["a2b2c2d1e2f2g2h2"], final_result["a1b1c2d2e2f2g2h2"])   
#             print(f"{circuit_name} on {backend_name} with {mitigation_tech} Completed!") 
            
#             #m3
#             if circuit_name != "GHZ_DC":
#                 print(f"Running {circuit_name} on {backend_name} with m3")    
#                 final_result_m3 = {'S': 0, "a1b1c1d1e1f1g1h1": 0, "a2b2c1d2e1f1g1h1": 0, "a2b2c2d1e2f2g2h2": 0, "a1b1c2d2e2f2g2h2": 0}

#                 mappings = [
#                     m3.utils.final_measurement_mapping(qc)
#                     for qc in isa_qc_list
#                 ]
#                 mit = m3.M3Mitigation(backend)
#                 mit.cals_from_system(mappings)
#                 quasis = [
#                     mit.apply_correction(res.data.meas.get_counts(), mapping)
#                     for res, mapping in zip(run_result, mappings)
#                 ]
#                 corrected_counts = [
#                     {k: round(v * 10000) for k, v in quasi.items()}
#                     for quasi in quasis
#                 ]
#                 for i, obs_key in zip(range(len(corrected_counts)) ,all_quantum_data_129["simulator"][circuit_name]["observables"].keys()):
#                     exp_value = exp_from_counts(corrected_counts[i])
#                     final_result_m3[obs_key] = exp_value
#                     final_result_m3['S'] += all_quantum_data_129["simulator"][circuit_name]["observables"][obs_key]["coeff"] * exp_value
            
#                 run_data = store_run_results(run_data, job.job_id(), available_service_name, circuit_name, backend_name, f"{mitigation_tech}+m3", final_result_m3["S"], final_result_m3["a1b1c1d1e1f1g1h1"],
#                                             final_result_m3["a2b2c1d2e1f1g1h1"], final_result_m3["a2b2c2d1e2f2g2h2"], final_result_m3["a1b1c2d2e2f2g2h2"]) 

#                 print(f"{circuit_name} on {backend_name} with {mitigation_tech}+m3 Completed!")

#             elif circuit_name == "GHZ_DC":
#                 print(f"Running {circuit_name} on {backend_name} with m3")    
#                 final_result_m3 = {'S': 0, "a1b1c1d1e1f1g1h1": 0, "a2b2c1d2e1f1g1h1": 0, "a2b2c2d1e2f2g2h2": 0, "a1b1c2d2e2f2g2h2": 0}

#                 mappings = [
#                     m3.utils.final_measurement_mapping(qc)
#                     for qc in isa_qc_list
#                 ]
#                 mit = m3.M3Mitigation(backend)
#                 mit.cals_from_system(mappings)
#                 quasis = [
#                     mit.apply_correction(res.join_data().get_counts(), mapping)
#                     for res, mapping in zip(run_result, mappings)
#                 ]
#                 corrected_counts_joined = [
#                     {k: round(v * 10000) for k, v in quasi.items()}
#                     for quasi in quasis
#                 ]

#                 corrected_counts = [
#                     marginal_counts(corrected_count_joined, indices = list(range(3,11)))
#                     for corrected_count_joined in corrected_counts_joined
#                 ]

#                 for i, obs_key in zip(range(len(corrected_counts)) ,all_quantum_data_129["simulator"][circuit_name]["observables"].keys()):
#                     exp_value = exp_from_counts(corrected_counts[i])
#                     final_result_m3[obs_key] = exp_value
#                     final_result_m3['S'] += all_quantum_data_129["simulator"][circuit_name]["observables"][obs_key]["coeff"] * exp_value
            
#                 run_data = store_run_results(run_data, job.job_id(), available_service_name, circuit_name, backend_name, f"{mitigation_tech}+m3", final_result_m3["S"], final_result_m3["a1b1c1d1e1f1g1h1"],
#                                             final_result_m3["a2b2c1d2e1f1g1h1"], final_result_m3["a2b2c2d1e2f2g2h2"], final_result_m3["a1b1c2d2e2f2g2h2"]) 

#                 print(f"{circuit_name} on {backend_name} with {mitigation_tech}+m3 Completed!")

#             run_data.to_csv(r"results\mid_run_data_saves.csv", index=False)

GHZ_UGC on ibm_fez with no_mitigation skipped!
GHZ_UGS on ibm_fez with no_mitigation skipped!
GHZ_SI on ibm_fez with no_mitigation skipped!
Available_serive : Vishal
Running GHZ_DC on ibm_fez with no_mitigation
Done Running GHZ_DC on ibm_fez. Took 12s
GHZ_DC on ibm_fez with no_mitigation Completed!
Running GHZ_DC on ibm_fez with m3
GHZ_DC on ibm_fez with no_mitigation+m3 Completed!
GHZ_UGC on ibm_marrakesh with no_mitigation skipped!
GHZ_UGS on ibm_marrakesh with no_mitigation skipped!
GHZ_SI on ibm_marrakesh with no_mitigation skipped!
GHZ_DC on ibm_marrakesh with no_mitigation skipped!
GHZ_UGC on ibm_fez with full_mitigation skipped!
GHZ_UGS on ibm_fez with full_mitigation skipped!
GHZ_SI on ibm_fez with full_mitigation skipped!
Available_serive : Vishal
GHZ_UGC on ibm_marrakesh with full_mitigation skipped!
GHZ_UGS on ibm_marrakesh with full_mitigation skipped!
GHZ_SI on ibm_marrakesh with full_mitigation skipped!
Available_serive : Vishal


Only for GHZ_DC with full mitigation

In [97]:
for mitigation_tech in sampler_options_list.keys():
    for backend_name in backend_list:
        for circuit_name in all_quantum_data_129["simulator"].keys():
            if (get_run_completion(mitigation_tech, backend_name, circuit_name) == False) or (circuit_name not in circuit_list):
                print(f"{circuit_name} on {backend_name} with {mitigation_tech} skipped!")
                continue

            available_service_name = safety_check(circuit_name, mitigation_tech, backend_name)
            if available_service_name == None:
                raise(Exception("NO SERVICE AVAILABLE!!!"))
            available_service = QiskitRuntimeService(name = available_service_name)
            print(f"Available_serive : {next(k for k, v in static_service.saved_accounts().items() if v == static_service.active_account())}")
            backend = available_service.backend(name = backend_name)
            if backend == None:
                raise(Exception("No Backend Available!!!"))
                       
            pm_lv3 = generate_preset_pass_manager(optimization_level=3, backend=backend, scheduling_method=None, seed_transpiler=42)

            if circuit_name == "GHZ_DC" and mitigation_tech=="full_mitigation":
                sampler = SamplerV2(mode = backend, options = sampler_options_list["no_mitigation"])
                
                run_circuits = []
                for qc in all_quantum_data_129["simulator"][circuit_name]["run_circuits"]:
                    template, outputs = apply_full_mitigation_dc(qc = qc, backend = backend, preset_pm = pm_lv3, num_dd = 1, num_randomizations = 32)
                    run_circuits.append([template, outputs])

                with open(r"data\run_circuits_dc.pkl", "wb") as f:
                    pickle.dump(run_circuits, f)

                pubs = [(run_ckt[0], run_ckt[1]["parameter_values"]) for run_ckt in run_circuits]
            
            else:
                continue

            print(f"Running {circuit_name} on {backend_name} with {mitigation_tech}")    
            try:
                job = sampler.run(pubs, shots = 10000)  
                run_result = job.result()
            except Exception as e:
                raise Exception("Job failed") from e
                
            print(f"Done Running {circuit_name} on {backend_name}. Took {job.metrics()["usage"]["quantum_seconds"]}s")
            update_run_completion(False, mitigation_tech, backend_name, circuit_name, job)

            final_result = {'S': 0, "a1b1c1d1e1f1g1h1": 0, "a2b2c1d2e1f1g1h1": 0, "a2b2c2d1e2f2g2h2": 0, "a1b1c2d2e2f2g2h2": 0}

            if circuit_name == "GHZ_DC" and mitigation_tech=="full_mitigation":
                    for k in range(4):  
                        res = run_result[k]
                        obs_key = list(all_quantum_data_129["simulator"][circuit_name]["observables"].keys())[k]
                        meas_data = res.data["meas"]
                        meas_data ^= BitArray.from_bool_array(run_circuits[k][1]["measurement_flips.meas"], "little")

                        total_exp = 0
                        for i in range(32):
                            counts = meas_data[i].get_counts()
                            exp_val = exp_from_counts(counts)
                            total_exp+=exp_val

                        total_exp = total_exp/32
                        final_result[obs_key] = total_exp
                        final_result['S'] += all_quantum_data_129["simulator"][circuit_name]["observables"][obs_key]["coeff"] * total_exp

            run_data = store_run_results(run_data, job.job_id(), available_service_name, circuit_name, backend_name, mitigation_tech, final_result["S"], final_result["a1b1c1d1e1f1g1h1"],
                                        final_result["a2b2c1d2e1f1g1h1"], final_result["a2b2c2d1e2f2g2h2"], final_result["a1b1c2d2e2f2g2h2"])   
            print(f"{circuit_name} on {backend_name} with {mitigation_tech} Completed!") 
            
            #m3
            if circuit_name == "GHZ_DC" and mitigation_tech=="full_mitigation":
                print(f"Running {circuit_name} on {backend_name} with m3")    
                final_result_m3 = {'S': 0, "a1b1c1d1e1f1g1h1": 0, "a2b2c1d2e1f1g1h1": 0, "a2b2c2d1e2f2g2h2": 0, "a1b1c2d2e2f2g2h2": 0}
                mit = m3.M3Mitigation(backend)

                for k in range(4):
                    result = copy.deepcopy(run_result[k])
                    joined_bits_original = result.join_data()

                    # Create a new list with XOR-corrected BitArrays
                    joined_bits = []

                    for i in range(32):
                        # Get the flip pattern
                        flip_pattern = run_circuits[k][1]["measurement_flips.meas"][i]
                        flip_array = np.array(flip_pattern, dtype=bool)
                        
                        # Calculate padding size
                        total_bits = joined_bits_original[i].num_bits
                        meas_bits = flip_array.shape[1]
                        cr_syn_bits = total_bits - meas_bits
                        
                        # Create padded flip
                        num_shots = flip_array.shape[0]
                        padding = np.zeros((num_shots, cr_syn_bits), dtype=bool)
                        padded_flip = np.concatenate([padding, flip_array], axis=1)
                        
                        # Apply XOR and append to new list
                        flip_bitarray = BitArray.from_bool_array(padded_flip, "little")
                        corrected_bitarray = joined_bits_original[i] ^ flip_bitarray
                        joined_bits.append(corrected_bitarray)

                    counts_dc = [
                        joined_bits[i].get_counts()
                        for i in range(32)
                    ]
                    mappings_dc = m3.utils.final_measurement_mapping(run_circuits[k][0])
                    mit.cals_from_system(mappings_dc)
                    quasis = [
                        mit.apply_correction(counts, mappings_dc)
                        for counts in counts_dc
                    ]
                    corrected_counts_dc = [
                        {k: round(v * 10000) for k, v in quasi.items()}
                        for quasi in quasis
                    ]

                    final_corrected_counts = [
                        marginal_counts(corrected_counts_dc[i], indices = list(range(3,11)))
                        for i in range(32)
                    ]

                    obs_key = list(all_quantum_data_129["simulator"][circuit_name]["observables"].keys())[k]

                    total_exp = 0
                    for i in range(32):
                        exp_val = exp_from_counts(final_corrected_counts[i])
                        total_exp+=exp_val

                    total_exp = total_exp/32
                    final_result_m3[obs_key] = total_exp
                    final_result_m3['S'] += all_quantum_data_129["simulator"][circuit_name]["observables"][obs_key]["coeff"] * total_exp
            
                run_data = store_run_results(run_data, job.job_id(), available_service_name, circuit_name, backend_name, f"{mitigation_tech}+m3", final_result_m3["S"], final_result_m3["a1b1c1d1e1f1g1h1"],
                                            final_result_m3["a2b2c1d2e1f1g1h1"], final_result_m3["a2b2c2d1e2f2g2h2"], final_result_m3["a1b1c2d2e2f2g2h2"]) 

                print(f"{circuit_name} on {backend_name} with {mitigation_tech}+m3 Completed!")

            run_data.to_csv(r"results\mid_run_data_saves.csv", index=False)

GHZ_UGC on ibm_fez with no_mitigation skipped!
GHZ_UGS on ibm_fez with no_mitigation skipped!
GHZ_SI on ibm_fez with no_mitigation skipped!
GHZ_DC on ibm_fez with no_mitigation skipped!
GHZ_UGC on ibm_marrakesh with no_mitigation skipped!
GHZ_UGS on ibm_marrakesh with no_mitigation skipped!
GHZ_SI on ibm_marrakesh with no_mitigation skipped!
GHZ_DC on ibm_marrakesh with no_mitigation skipped!
GHZ_UGC on ibm_fez with full_mitigation skipped!
GHZ_UGS on ibm_fez with full_mitigation skipped!
GHZ_SI on ibm_fez with full_mitigation skipped!
GHZ_DC on ibm_fez with full_mitigation skipped!
GHZ_UGC on ibm_marrakesh with full_mitigation skipped!
GHZ_UGS on ibm_marrakesh with full_mitigation skipped!
GHZ_SI on ibm_marrakesh with full_mitigation skipped!
Available_serive : Vishal
Running GHZ_DC on ibm_marrakesh with full_mitigation
Done Running GHZ_DC on ibm_marrakesh. Took 348s
GHZ_DC on ibm_marrakesh with full_mitigation Completed!
Running GHZ_DC on ibm_marrakesh with m3
GHZ_DC on ibm_marrakesh

In [98]:
display(run_data)

,date,time,job_id,service_acc,circuit,mitigation,backend,S,a1b1c1d1e1f1g1h1,a2b2c1d2e1f1g1h1,a2b2c2d1e2f2g2h2,a1b1c2d2e2f2g2h2,Ideal S,Ideal_a1b1c1d1e1f1g1h1,Ideal_a2b2c1d2e1f1g1h1,Ideal_a2b2c2d1e2f2g2h2,Ideal_a1b1c2d2e2f2g2h2
0,19-01-2026,14:50:34,d5mvdmbh36vs73bilvc0,swapnil_charutarhealth,GHZ_UGC,no_mitigation,ibm_fez,2.983600,0.743200,-0.731200,0.748800,0.760400,4,1,-1,1,1
1,19-01-2026,14:51:22,d5mvdmbh36vs73bilvc0,swapnil_charutarhealth,GHZ_UGC,no_mitigation+m3,ibm_fez,3.380737,0.837984,-0.830932,0.848248,0.863573,4,1,-1,1,1
2,19-01-2026,17:33:47,d5n0bet9j2ac739lc5ug,swapnil_charutarhealth,GHZ_UGC,full_mitigation,ibm_marrakesh,2.980800,0.730200,-0.759000,0.740600,0.751000,4,1,-1,1,1
3,19-01-2026,17:57:52,d5n0bet9j2ac739lc5ug,swapnil_charutarhealth,GHZ_UGC,full_mitigation+m3,ibm_marrakesh,3.326079,0.807723,-0.844216,0.815455,0.858685,4,1,-1,1,1
4,19-01-2026,18:10:44,d5n0a1ph2mqc739bdl70,swapnil_charutarhealth,GHZ_DC,full_mitigation,ibm_fez,1.409400,0.345800,-0.355800,0.340800,0.367000,4,1,-1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,31-01-2026,12:58:26,d5uqtspfodos73emompg,vedant,GHZ_DC,no_mitigation+m3,ibm_fez,0.930062,0.224372,-0.241414,0.232942,0.231334,4,1,-1,1,1
68,31-01-2026,13:28:51,d5ur0vscqoec73dlb240,vedant,GHZ_DC,full_mitigation,ibm_fez,1.760637,0.429781,-0.434769,0.458806,0.437281,4,1,-1,1,1
69,31-01-2026,13:32:20,d5ur0vscqoec73dlb240,Vishal,GHZ_DC,full_mitigation+m3,ibm_fez,2.037202,0.499044,-0.502840,0.532302,0.503017,4,1,-1,1,1
70,31-01-2026,13:43:57,d5urge4cqoec73dlbkpg,Vishal,GHZ_DC,full_mitigation,ibm_marrakesh,2.528106,0.626100,-0.627331,0.637825,0.636850,4,1,-1,1,1


In [99]:
run_data.to_csv(r"results\Run_data.csv", index=False)